In [3]:
# 日本語対応のモジュールをインストールする（その１）
!pip install japanize-matplotlib
# 日本語対応のモジュールをインストールする（その２）
!apt-get install -y fonts-ipafont-gothic
# janomeをインストールする
%pip install janome

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-ipafont-gothic is already the newest version (00303-21ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 57.0 MB/s eta 0:00:00


In [4]:
# 各ライブラリーをインポートする
import re
import numpy as np
import pandas as pd
from janome.tokenizer import Tokenizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report

# 乱数シードを固定
RANDOM_STATE = 42

In [5]:
# Inside Airbnb 東京の listings.csv(詳細版を推奨)
DATA_PATH = "listings.csv"
df = pd.read_csv(DATA_PATH)
print("行数・列数:", df.shape)
print("先頭の列:", list(df.columns)[:15], "...")

行数・列数: (453, 85)
先頭の列: ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since'] ...


In [6]:
# 詳細版には description / neighborhood_overview がある。存在する列だけ結合する。
TEXT_CANDIDATES = ["name", "description", "neighborhood_overview"]
text_cols = [c for c in TEXT_CANDIDATES if c in df.columns]
print("テキストとして使う列:", text_cols)

def strip_html(s: str) -> str:
    """Airbnb の説明文には <br /> などのタグや実体参照が混ざるため除去する。"""
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"&[a-zA-Z]+;", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

df["text"] = ""
for c in text_cols:
    df["text"] = df["text"] + " " + df[c].fillna("").astype(str)

df["text"] = df["text"].map(strip_html)

# エリア列(区)。詳細版は neighbourhood_cleansed、要約版は neighbourhood。
if "neighbourhood_cleansed" in df.columns:
    AREA_COL = "neighbourhood_cleansed"
elif "neighbourhood" in df.columns:
    AREA_COL = "neighbourhood"
else:
    AREA_COL = None
print("エリア列:", AREA_COL)

テキストとして使う列: ['name', 'description', 'neighborhood_overview']
エリア列: neighbourhood_cleansed


In [8]:
_tokenizer = Tokenizer()
TARGET_POS = {"名詞","形容詞"}
# 内容語のみ採用(動詞は課題で追加)
DROP_POS_SUB = {"数","代名詞","非自立","接尾","副詞可能"}

# ストップワード(汎用語・Airbnb頻出のノイズ語)。各自で拡張してよい。
STOPWORDS = {
    "こと", "もの", "ため", "よう", "それ", "これ", "ここ", "そこ", "とき", "中", "方", "等", "様", "為", "他", "上", "下",
    "前", "後", "際", "部屋", "お部屋", "物件", "ゲスト", "滞在", "利用", "ホスト",
    "room", "house", "apartment", "place", "stay", "guest", "host", "tokyo", "japan", "min", "station",
}

def tokenize_ja(text: str):
    """名詞・形容詞を原形で抽出し、1文字語・数字・ストップワードを除く。"""
    out = []
    for tok in _tokenizer.tokenize(text):
        parts = tok.part_of_speech.split(",")
        pos_major = parts[0]
        pos_sub = parts[1] if len(parts) > 1 else ""

        if pos_major not in TARGET_POS:
            continue
        if pos_sub in DROP_POS_SUB:
            continue

        base = tok.base_form if tok.base_form != "*" else tok.surface
        base = base.strip().lower()

        if len(base) <= 1:
            continue
        if re.fullmatch(r"[0-9]+", base): # 純粋な数字は除外
            continue
        if base in STOPWORDS:
            continue

        out.append(base)
    return out
# 英語 vs 日本語の対比デモ
print("\n--- 分かち書きデモ ---")
print("英語(空白分割):", "Cozy room near Shibuya station, free WiFi".split())
print("日本語(分かち書き):", tokenize_ja("渋谷駅まで徒歩5分の清潔で明るいお部屋。無料WiFi完備で観光に便利です。"))


--- 分かち書きデモ ---
英語(空白分割): ['Cozy', 'room', 'near', 'Shibuya', 'station,', 'free', 'WiFi']
日本語(分かち書き): ['渋谷', '徒歩', '清潔', '明るい', '無料', 'wifi', '完備', '観光', '便利']


In [9]:
# 形態素解析は重いので「先に1回だけ」分かち書きし、空白区切り文字列に変換しておく。
SAMPLE_SIZE = 3000
work = df.copy() if SAMPLE_SIZE is None else df.sample(
    n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE
).copy()

work = work[work["text"].str.len() > 0].copy()
work["tokens"] = work["text"].map(tokenize_ja)
work["joined"] = work["tokens"].map(lambda t: " ".join(t))
work = work[work["joined"].str.len() > 0].copy()

print("\n対象文書数:", len(work))
print("平均トークン数:", round(work["tokens"].map(len).mean(), 1))


対象文書数: 453
平均トークン数: 62.8


In [10]:
# 分かち書き済みなので token_pattern=r"\S+" で空白区切りをそのまま語にする。
count_vec = CountVectorizer(token_pattern=r"\S+", lowercase=False, min_df=5, max_df=0.5)
X_count = count_vec.fit_transform(work["joined"])

tfidf_vec = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, min_df=5, max_df=0.5)
X_tfidf = tfidf_vec.fit_transform(work["joined"])

vocab = np.array(count_vec.get_feature_names_out())

print("\nBoW 行列:", X_count.shape, "語彙数:", len(vocab))
sparsity = 1.0 - X_count.nnz / (X_count.shape[0] * X_count.shape[1])
print(f"疎性(ゼロ要素の割合): {sparsity:.4f}")

# 高頻度語と高TF-IDF語の違い
freq = np.asarray(X_count.sum(axis=0)).ravel()
top_freq = vocab[np.argsort(freq)[::-1][:20]]
print("高頻度語 Top20:", list(top_freq))

tfidf_sum = np.asarray(X_tfidf.sum(axis=0)).ravel()
top_tfidf = vocab[np.argsort(tfidf_sum)[::-1][:20]]
print("高TF-IDF語 Top20:", list(top_tfidf))


BoW 行列: (453, 773) 語彙数: 773
疎性(ゼロ要素の割合): 0.9489
高頻度語 Top20: ['is', 'you', 'home', 'bedroom', 'located', 'from', 'downtown', 'parking', 'your', 'private', 'or', 'kitchen', 'on', 'street', 'cozy', 'restaurants', 'enjoy', 'center', 'free', 'space']
高TF-IDF語 Top20: ['is', 'home', 'you', 'bedroom', 'from', 'located', 'downtown', 'your', 'private', 'parking', 'cozy', 'or', 'enjoy', 'street', 'on', 'restaurants', 'center', 'kitchen', 'free', 'space']


In [11]:
if AREA_COL is not None:
    print("\n--- エリア別 特徴語(TF-IDF平均上位) ---")
    top_areas = work[AREA_COL].value_counts().head(6).index.tolist()
    for area in top_areas:
        mask = (work[AREA_COL] == area).values
        if mask.sum() < 10:
            continue
        mean_tfidf = np.asarray(X_tfidf[mask].mean(axis=0)).ravel()
        top = vocab[np.argsort(mean_tfidf)[::-1][:10]]
        print(f"[{area}] ({mask.sum()}件): " + " / ".join(top))
else:
    print("エリア列が無いためスキップ")


--- エリア別 特徴語(TF-IDF平均上位) ---
[SIXTH WARD] (106件): from / park / center / square / washington / state / heart / historic / street / is
[TENTH WARD] (49件): bedroom / private / is / new / hospitals / st / home / located / queen / med
[SECOND WARD] (43件): state / downtown / historic / street / apt / center / plaza / empire / mvp / you
[THIRTEENTH WARD] (39件): is / home / renovated / our / bedroom / bed / located / floor / mi / centrally
[NINTH WARD] (36件): private / home / on / shared / no / suite / victorian / located / have / bedroom
[THIRD WARD] (33件): downtown / located / is / palace / newly / from / all / bedroom / city / nys


In [12]:
# 重要: LDAは「出現回数(カウント)」を入力にする。TF-IDFではない。
N_TOPICS = 8
lda = LatentDirichletAllocation(
    n_components=N_TOPICS, learning_method="batch",
    max_iter=20, random_state=RANDOM_STATE,
)
doc_topic = lda.fit_transform(X_count) # 文書×トピックの確率分布

def show_topics(model, feature_names, n_top=10):
    for k, comp in enumerate(model.components_):
        top = feature_names[np.argsort(comp)[::-1][:n_top]]
        print(f"トピック {k:2d}: " + " / ".join(top))

print(f"\n--- LDA トピック (k={N_TOPICS}) ---")
show_topics(lda, vocab, n_top=10)

work["topic"] = doc_topic.argmax(axis=1)
print("\nトピック別の文書数:")
print(work["topic"].value_counts().sort_index().to_string())

# トピック数 kの比較
print("\n--- トピック数 k の比較 (perplexity) ---")
for k in [4, 6, 8, 10, 12]:
    m = LatentDirichletAllocation(
        n_components=k, learning_method="batch", max_iter=20, random_state=RANDOM_STATE
    ).fit(X_count)
    print(f"k={k:2d} perplexity={m.perplexity(X_count):.1f}")


--- LDA トピック (k=8) ---
トピック  0: parking / free / fi / wi / br / fully / comfy / it / kitchen / equipped
トピック  1: parking / is / friendly / space / off / street / pet / entry / floor / tv
トピック  2: on / private / is / queen / bedroom / bed / kitchen / large / new / home
トピック  3: home / kitchen / at / bedroom / located / family / bath / fully / bathroom / you
トピック  4: from / is / bedroom / minutes / you / home / restaurants / located / street / or
トピック  5: your / you / cozy / downtown / breakfast / convenience / business / home / perfect / re
トピック  6: state / historic / empire / plaza / is / downtown / renovated / unit / apt / bed
トピック  7: center / from / is / medical / square / modern / home / an / st / garden

トピック別の文書数:
topic
0     18
1     23
2     73
3     52
4    102
5     30
6     86
7     69

--- トピック数 k の比較 (perplexity) ---
k= 4 perplexity=431.3
k= 6 perplexity=423.9
k= 8 perplexity=423.9
k=10 perplexity=421.8
k=12 perplexity=429.2


In [13]:
print("\n--- コサイン類似度による類似文書検索 ---")
query_idx = 0
sims = cosine_similarity(X_tfidf[query_idx], X_tfidf).ravel()
order = np.argsort(sims)[::-1]
order = order[order != query_idx][:5]

name_col = "name" if "name" in work.columns else "text"
print("クエリ:", str(work.iloc[query_idx][name_col])[:60])
for idx in order:
    print(f" 類似度 {sims[idx]:.3f} : " + str(work.iloc[idx][name_col])[:60])


--- コサイン類似度による類似文書検索 ---
クエリ: Downtown ALB • Hot Tub • Game Room • Free Parking
 類似度 0.285 : 4 bedroom house in Albany, 4 minutes from AMC
 類似度 0.281 : ★ BEAUTIFUL  2 BED 2 BATH  w/ SOAKER TUB + W/D★
 類似度 0.278 : Downtown Albany 1 Bed + Workstation @ Maiden Lane
 類似度 0.263 : Downtown Albany 2 Bedroom + Workstation @ The Mark
 類似度 0.235 : Central cozy 4 bedroom w/King bed


In [14]:
print("\n--- k-means クラスタ × LDA トピック の対応 ---")
km = KMeans(n_clusters=N_TOPICS, n_init=10, random_state=RANDOM_STATE)
work["cluster"] = km.fit_predict(X_tfidf)
print(pd.crosstab(work["cluster"], work["topic"]).to_string())


--- k-means クラスタ × LDA トピック の対応 ---
topic    0   1   2   3   4   5   6   7
cluster                               
0        1   2  20  11   9   0   4   2
1        0   0   0   0   0  25   0   0
2        4  12  10  31  49   4   5  23
3        0   6  14   3   6   0   6   4
4        8   0   0   2   2   0  18   1
5        5   3   6   1  33   1  35  39
6        0   0   0   0   0   0  12   0
7        0   0  23   4   3   0   6   0


In [16]:
print("\n--- テキスト分類 ---")
if "review_scores_rating" in work.columns and pd.to_numeric(work["review_scores_rating"], errors="coerce").notna().sum() > 50:
    r = pd.to_numeric(work["review_scores_rating"], errors="coerce")
    work2 = work.assign(_rating=r).dropna(subset=["_rating"]).copy()
    thr = work2["_rating"].quantile(0.5) # 中央値で2分割
    y = (work2["_rating"] >= thr).astype(int).values
    target_name = f"高評価(>= {thr:.2f}) vs 低評価"
elif "room_type" in work.columns:
    work2 = work[work["room_type"].notna()].copy()
    y = (work2["room_type"] == "Entire home/apt").astype(int).values
    target_name = "Entire home/apt vs その他"
else:
    work2 = None
    y = None
    print("分類に使えるターゲット列が無いためスキップ")

if work2 is not None:
    Xc = count_vec.transform(work2["joined"]) # 同じ語彙で変換
    print("タスク:", target_name, "正例率:", round(float(y.mean()), 3))

    X_tr, X_te, y_tr, y_te = train_test_split(
        Xc, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

    clf = MultinomialNB()
    clf.fit(X_tr, y_tr)
    print(classification_report(y_te, clf.predict(X_te), digits=3))

    # 第7-8回の復習：層化K分割で安定評価
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    f1 = cross_val_score(clf, Xc, y, cv=cv, scoring="f1")
    print("F1 (5-fold):", f1.round(3), "平均:", round(float(f1.mean()), 3))

    # どの語がどちらのクラスを示すか（ナイーブベイズの対数確率差）
    fn = np.array(count_vec.get_feature_names_out())
    diff = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
    print("正例を示す語 Top15:", list(fn[np.argsort(diff)[::-1][:15]]))
    print("負例を示す語 Top15:", list(fn[np.argsort(diff)[:15]]))


--- テキスト分類 ---
タスク: 高評価(>= 4.85) vs 低評価 正例率: 0.522
              precision    recall  f1-score   support

           0      0.717     0.702     0.710        47
           1      0.736     0.750     0.743        52

    accuracy                          0.727        99
   macro avg      0.727     0.726     0.726        99
weighted avg      0.727     0.727     0.727        99

F1 (5-fold): [0.707 0.575 0.628 0.729 0.643] 平均: 0.656
正例を示す語 Top15: ['closet', 'tub', 'stainless', 'hybrid', 'outdoor', 'abba', 'there', 'jesse', 'vaping', 'comply', 'buel', 'strict', 'plan', 'grand', 'cannot']
負例を示す語 Top15: ['pastures', 'me', 'courts', 'regional', 'anytime', 'earl', 'older', 'affordable', 'save', 'dove', 'bd', 'furnishings', 'non', 'leisure', 'mads']
